# 실습 12 · 가상 스크리닝 (Virtual Screening)
### 수만 개 화합물에서 유망 후보를 걸러내는 실전 파이프라인

**왜 중요한가 (현장 맥락)**
제약사는 **수십만~수백만 개** 화합물 라이브러리를 보유합니다.
전부 실험할 수 없으니, 컴퓨터로 **유망 후보만 선별(가상 스크리닝)** 합니다.
대표 전략 2가지를 실전 파이프라인으로 구현합니다:
1. **리간드 기반**: 알려진 활성물질과 **닮은** 분자 찾기 (유사도 검색)
2. **약물유사성/ADMET 필터**: 약이 되기 어려운 분자 제거

> 예제 10(유사도)·11(QSAR)의 기본기를 **하나의 스크리닝 워크플로우**로 통합합니다.


In [ ]:
!pip install rdkit -q
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Crippen, DataStructs, Draw

# 스크리닝 대상 라이브러리로 실제 BACE 데이터의 SMILES 사용 (1500여 개)
url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/bace.csv"
lib = pd.read_csv(url)[["mol","pIC50"]].rename(columns={"mol":"smiles"})
print("스크리닝 대상 화합물:", len(lib), "개")
lib.head()

## 1단계 · 리간드 기반 유사도 검색
알려진 강력한 저해제(**query**)와 구조가 닮은 분자를 찾습니다.
"비슷한 구조 → 비슷한 활성" 가정에 기반합니다.


In [ ]:
# 지문 계산 함수
def get_fp(smi):
    m = Chem.MolFromSmiles(smi)
    return AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048) if m else None

lib["fp"] = lib["smiles"].apply(get_fp)
lib = lib[lib["fp"].notnull()].reset_index(drop=True)

# query: 데이터에서 가장 강력한(pIC50 최대) 알려진 저해제를 기준물질로
query_row = lib.loc[lib["pIC50"].idxmax()]
query_fp = query_row["fp"]
print("기준물질 pIC50:", round(query_row["pIC50"],2))

# 라이브러리 전체와 Tanimoto 유사도 계산
lib["유사도"] = lib["fp"].apply(lambda f: DataStructs.TanimotoSimilarity(query_fp, f))
top = lib.sort_values("유사도", ascending=False).head(20)
print("\n기준물질과 가장 닮은 분자 top 5:")
print(top[["smiles","pIC50","유사도"]].head().round(3).to_string(index=False))

In [ ]:
# 유사도와 실제 활성의 관계 — '닮으면 활성도 높은가?' 검증
plt.figure(figsize=(7,4.5))
plt.scatter(lib["유사도"], lib["pIC50"], alpha=0.3)
plt.xlabel("기준물질과의 Tanimoto 유사도"); plt.ylabel("실제 pIC50")
plt.title("유사도 ↔ 활성 관계 (오른쪽 위 = 닮고 강력)")
corr = lib["유사도"].corr(lib["pIC50"])
plt.text(0.05,0.9,f"상관계수={corr:.2f}",transform=plt.gca().transAxes)
plt.show()

## 2단계 · 약물유사성/ADMET 필터
유사도 상위 후보 중 **약이 되기 좋은 성질**을 가진 것만 남깁니다 (Lipinski + 기본 ADMET).


In [ ]:
def drug_filter(smi):
    m = Chem.MolFromSmiles(smi)
    mw, logp = Descriptors.MolWt(m), Crippen.MolLogP(m)
    hbd, hba = Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m)
    tpsa = Descriptors.TPSA(m); rot = Descriptors.NumRotatableBonds(m)
    # 통과 조건 (경구약 경험칙)
    ok = (mw<=500) and (logp<=5) and (hbd<=5) and (hba<=10) and (tpsa<=140) and (rot<=10)
    return pd.Series({"분자량":round(mw,1),"LogP":round(logp,2),
                      "TPSA":round(tpsa,1),"통과":ok})

cand = top.copy()
props = cand["smiles"].apply(drug_filter)
cand = pd.concat([cand.reset_index(drop=True), props], axis=1)
passed = cand[cand["통과"]]
print(f"유사도 top20 중 약물유사성 통과: {len(passed)}개")
passed[["smiles","pIC50","유사도","분자량","LogP","TPSA"]].round(3)

## 3단계 · 최종 후보 시각화
유사도 높고 + 약물유사성 통과한 **최종 후보 분자**를 구조로 확인합니다.


In [ ]:
final = passed.head(6)
mols = [Chem.MolFromSmiles(s) for s in final["smiles"]]
legs = [f"pIC50={p:.1f}\nsim={s:.2f}" for p,s in zip(final["pIC50"], final["유사도"])]
Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(230,190), legends=legs)

## 4단계 · 스크리닝 깔때기(funnel) 요약
가상 스크리닝은 **단계적으로 후보를 좁히는 깔때기**입니다.


In [ ]:
stages = ["전체 라이브러리", "유사도 top20", "약물유사성 통과", "최종 후보"]
counts = [len(lib), len(top), len(passed), len(final)]
plt.figure(figsize=(7,4))
plt.barh(stages[::-1], counts[::-1], color="#4C72B0")
for i,c in enumerate(counts[::-1]):
    plt.text(c, i, f" {c}", va="center")
plt.xscale("log"); plt.xlabel("화합물 수 (로그 스케일)")
plt.title("가상 스크리닝 깔때기"); plt.tight_layout(); plt.show()

## 정리 & 현장 응용
- **유사도 검색(리간드 기반)** → **ADMET/Lipinski 필터** → 최종 후보 선별
- 여기에 예제 11의 **QSAR 활성 예측**을 추가하면 더 강력한 3중 필터
- 다음 단계(예제 13·14): 선별된 후보를 **표적 단백질에 도킹**해 결합력 확인
- 실무 확장: 상용 라이브러리(Enamine REAL 수십억 개)를 이런 필터로 좁힘
